In [1]:
from handlers.uni_handlers.base import get_uni_handler

uni_handler = get_uni_handler('northumbria')

In [2]:
northumbria = uni_handler()

In [5]:
courses = northumbria.get_courses()

In [30]:
from agents.utils import run_multithreaded_handler
import json
import os
from settings import (
    university_data_dir, 
)



def get_requirements_countries(university_name="northumbria") -> dict:
    path = os.path.join(university_data_dir, university_name, "countries.json")
    if os.path.exists(path):
        return json.load(open(path))
    
    uni_handler = get_uni_handler(university_name)()
    countries = uni_handler.get_countries()
    json.dump(countries, open(path, 'w'), indent=4)
    
    return countries


def get_requirements_courses(university_name="northumbria") -> dict:
    path = os.path.join(university_data_dir, university_name, "courses.json")
    if os.path.exists(path):
        return json.load(open(path))
    
    uni_handler = get_uni_handler(university_name)()
    courses = uni_handler.get_courses()
    json.dump(courses, open(path, 'w'), indent=4)
    
    return courses


def extract_country_x_course_requirements(uni_name="northumbria"):
    # parse uni req
    # get all courses and countries

    f_name = os.path.join(university_data_dir, uni_name, "course_x_country_requirements.json")
    if os.path.exists(f_name):
        print("University requirements already extracted. Skipping...")
        return 
    
    course_x_country_requirements = dict()
    
    requirements_courses: dict = get_requirements_courses(uni_name)
    requirements_countries: dict = get_requirements_countries(uni_name)
    courses = requirements_courses.keys()
    countries = requirements_countries.keys()

    prompt_get_course_req = """
    given the requirements for entry into university below, 
    extract and summarize in very short  the fields structurally in json isolating - academic, english, work experience, documents needed and other requirements separately . 
    if there are multiple levels or types of sub courses, isolate them as well. Dont output anything else but the json and dont make up any requirement that isnt mentioned. 
    Provide any notes you have in a separate column in the json.
    here is the requirements - {course_requirement}
    """

    prompt_get_country_req = """
    You are given the requirements for entry into university below for a student from UK, 
    and rules below for a student applying from a different country - {country}, recreate above json for university requirements for same course 
    if student is from {country} by correctly mapping and altering requirements given based on the rules given . 
    Dont output anything else but the json and dont make up any requirement that isnt mentioned. 
    Provide any notes you have in a separate column in the json about the replacements in entry requirements you made.
    here are the rules for a student applying from {country} - {country_requirement}
    """

    system_prompt = "You are an expert in extracting summarized information in structured json format from unstructured text"

    course_prompts = {
        course: prompt_get_course_req.format(course_requirement=requirements_courses[course])
        for course in courses
    }
    course_responses = run_multithreaded_handler(
        course_prompts, ordered=True, system_prompt=system_prompt
    )

    for course, course_response in course_responses.items():
        country_prompts = {
            country:
            prompt_get_country_req.format(
                country=country, country_requirement=requirements_countries[country]
            ) + course_response
            for country in countries if country in ["India", "Bangladesh"]
        }

        country_responses = run_multithreaded_handler(
            country_prompts, ordered=True, system_prompt=system_prompt
        )

        for country, country_response in country_responses.items():
            if country not in course_x_country_requirements:
                course_x_country_requirements[country] = dict()
            course_x_country_requirements[country][course] = (course_response, country_response)

    json.dump(
        course_x_country_requirements, 
        open(f_name, 'w'), 
        indent=4
    )


def get_university_requirements(university_name, course, country):
    extract_country_x_course_requirements(uni_name=university_name)
    uni_reqs = json.load(open(os.path.join(university_data_dir, university_name, "course_x_country_requirements.json")))
    uni_req = uni_reqs[country][course][0].split("```")[1]
    return uni_req

In [31]:
extract_country_x_course_requirements()

University requirements already extracted. Skipping...


In [32]:
from settings import (
    student_output_docs_dir, 
    student_data_dir,
    summary_folder
)


def get_student_doc_map(student_id):
    student_docs_folder = os.path.join(student_data_dir, student_id, student_output_docs_dir)
    student_doc_map = json.load(open(os.path.join(student_data_dir, "student_doc_map.json")))
    for f in os.listdir(student_docs_folder):
        if student_id not in student_doc_map:
            student_doc_map[student_id] = dict()

        if os.path.isfile(os.path.join(student_docs_folder, f)):
            if f not in student_doc_map[student_id]:
                student_doc_map[student_id][f] = len(student_doc_map[student_id]) + 1
    
    json.dump(student_doc_map, open(os.path.join(student_data_dir, "student_doc_map.json"), 'w'), indent=4)

    reverse = {v: k for k, v in student_doc_map[student_id].items()}
    json.dump(reverse, open(os.path.join(student_data_dir, "student_doc_map_inv.json"), 'w'), indent=4)
    
    return student_doc_map

In [33]:
def get_student_qualifications_data(student_id):
    student_doc_map = get_student_doc_map(student_id)
    student_docs_folder = os.path.join(student_data_dir, student_id, student_output_docs_dir)
    out_dir = os.path.join(student_docs_folder, summary_folder)
    os.makedirs(out_dir, exist_ok=True)

    summarization_prompt = \
    """
    Below is a text generated by an OCR model from parsing a PDF document. 

    The PDF is of the official documents of a student like certificates, passport and qualifications while applying to a university for masters or undergraduate programs. 
    Infer the name of the document and summarize the key details like qualifications, personal information for each of these that would be crucial in applying to international universities. 
    Clearly call out any ambiguous info and info that is not clearly mentioned. Do not make assumptions. Here is the text - 
    {student_doc_content}
    """


    file_summarization_prompts_dict = {
        os.path.join(out_dir, file_name): 
        f"{summarization_prompt} {open(os.path.join('.', student_docs_folder, file_name), encoding='utf8').read()}"
        for file_name in os.listdir(student_docs_folder)
        if not os.path.exists(os.path.join(out_dir, file_name)) \
            and os.path.isfile(os.path.join(student_docs_folder, file_name))\
            and file_name.endswith(".txt")
    }
    
    if not len(file_summarization_prompts_dict):
        print("Student documents already summarized. Skipping...")
    else:
        system_prompt = "You are an expert in summarizing broken text extracted from an ocr model on documents"
        file_contents_responses = run_multithreaded_handler(
            file_summarization_prompts_dict, ordered=True, system_prompt=system_prompt
        )

        for out_file_path, response in file_contents_responses.items():
            with open(f"{out_file_path}", 'w') as f:
                f.write(f"file_name: {os.path.basename(out_file_path)}\nContent: {response}")

    student_data = f"\n{'-'*20}\n{'-'*20}\n".join([
        f"Doc ID: {student_doc_map[student_id][file_name]}\n\n{open(os.path.join(out_dir, file_name)).read()}"
        for file_name in os.listdir(out_dir) if os.path.isfile(os.path.join(out_dir, file_name))
    ])
    return student_data


In [34]:
from agents.utils import get_llm_response


def process_student_qualifications(university_requirement, student_qual):
    def get_response():
        output_example = """
        {
            "requirement": "IELTS 6.5 (or above) with no single element below 5.5 or equivalent",
            "qualification": "No IELTS score mentioned",
            "qualification_result": "Does not meet the requirement",
            "doc_id": "1",
            "qualification_text": "No IELTS score mentioned",
        }
        """
        STUDENT_GET_KEYS_DATA_PROMPT = f"""
        Below are qualifications of a student given across all the documents he has submitted. Given these qualifications and some university requirements in key, value json format.
        Create a new JSON keeping the university key, value pairs intact, and add the following information.
        1. requirement: The requirement for the given key
        2. qualification: Student qualifications for the given key along with it to the same json.
        3. qualification_result: Clearly mention if the student qualifications does not have the qualifications corresponding to a key present in the updated json you provide. Make sure to provide only factual details of the student qualifications from the documents provided. 
        4. doc_id: Provide the document ID for the student document
        5. qualification_text: Provide the text extracted from the student document responsible for the qualification.

        
        Here are the student qualifications - \n {student_qual}. \n
        Here are the key, value json for university requirements for the country and course student has applied to - {university_requirement}.\n
        If the requirements have more than one course or levels of course, add student qualifications for each.
        Do not output any other information but the updated JSON and do not output any key, value pairs that is not about a university requirement criteria.

        Output Example:
        {output_example}
        """

        student_value_pairs = get_llm_response(
            STUDENT_GET_KEYS_DATA_PROMPT, 
            system_prompt="You are an expert in extracting information from texts in a given key, value pair format for a given set of keys"
        )
        print("Student Value Pairs Extracted")

        return student_value_pairs

    value_pairs = None
    while not value_pairs or '```' not in value_pairs:
        value_pairs = get_response()
    
    value_pairs = json.loads(value_pairs.split("```")[1])        

    return value_pairs


def get_processed_student_data(uni_name, course, country, student_id):
    university_requirements = get_university_requirements(
        uni_name, course, country
    )
    student_qualifications = get_student_qualifications_data(student_id)
    processed_student_qualifications = process_student_qualifications(
        university_requirements, 
        student_qualifications
    )
    
    return processed_student_qualifications

In [35]:
def decide_on_student_qualifications(student_value_pairs: dict):
    def get_response():
        output_example = """
        {
            "requirement": "IELTS 6.5 (or above) with no single element below 5.5 or equivalent",
            "qualification": "No IELTS score mentioned",
            "qualification_result": "Does not meet the requirement",
            "doc_id": "1",
            "qualification_text": "No IELTS score mentioned",
            "decision": "fail",
            "reasoning": "The student does not have an IELTS score, which does not meet the requirement of 6.5 or above.",
            "support_doc_id": "1",
            "supporting_text": "No IELTS score mentioned"
        }
        """

        GET_KEYS_DECISION_PROMPT = f"""
        Given a JSON showing university requirements for a given course, student qualifications for each student file, append a two new fields for each key.
        1. decision: The decision of if the student meets criteria or not. The decision can be pass, fail or ambiguous. Decision is to be ambiguous only if student qualifications for that university requirement arent clear.
        2. reasoning: The reasoning for the decision. The reasoning should be a short summary of why the student meets or fails to meet the criteria. If the decision is ambiguous, the reasoning should be a short summary of why the student qualifications are not clear.
        3. support_doc_id: Support the decision by giving the doc ID responsible for the qualification
        4. supporting_text: Support the decision by providing the text extracted from the student document responsible for the qualification. If the decision is ambiguous, mention the text that is missing or unclear.
        Do not output anything but the JSON. Here is the json with requirement, qualifications - {str(student_value_pairs)}

        Output Example:
        {output_example}
        """

        student_decision_pairs = get_llm_response(
            GET_KEYS_DECISION_PROMPT, 
            system_prompt="You are an expert in evaluating if a students qualifications meets a given university requirements both in terms of academics, english and subjective requirements by the university"
        )

        return student_decision_pairs
    
    decision_pairs = None
    while not decision_pairs or '```' not in decision_pairs:
        decision_pairs = get_response()
    print("Student Decision Pairs Extracted")

    return decision_pairs


def get_student_qualifications_decisions(uni_name, course, country, student_id):
    student_value_pairs = get_processed_student_data(uni_name, course, country, student_id)
    student_decisions = decide_on_student_qualifications(student_value_pairs)
    return student_decisions

In [37]:
university_name = "northumbria"
student_id = "10445379"
country = "Bangladesh"
course = "MSc Digital Marketing (with Advanced Practice)"

student_data = {
    'student_id': student_id,
    'first_name': "John",
    'last_name': "Doe",
    'email': "",
    'course': course,
    'country': country
}

decision_result = get_student_qualifications_decisions(
    uni_name=university_name, 
    course=course,
    country=country,
    student_id=student_id
)

University requirements already extracted. Skipping...
Student documents already summarized. Skipping...
Student Value Pairs Extracted
Student Decision Pairs Extracted
